# 🔌 Model Context Protocol (MCP) Integration with Azure AI Foundry (Python)

## 📋 MCP Enterprise Integration Tutorial

This notebook demonstrates advanced **Model Context Protocol (MCP)** integration using Azure AI Foundry and the Microsoft Agent Framework. You'll learn how to connect AI agents to external data sources and services through standardized MCP interfaces for enterprise-grade integrations.

## 🎯 Learning Objectives

### 🔗 **MCP Protocol Understanding**
- **Protocol Architecture**: Learn the MCP specification and communication patterns
- **Server Integration**: Connect to MCP-enabled services and data sources
- **Streaming Operations**: Handle real-time data streams through MCP
- **Tool Orchestration**: Coordinate multiple MCP tools within agent workflows

### 🏢 **Enterprise MCP Applications**
- **Knowledge Base Integration**: Connect agents to corporate knowledge repositories
- **API Service Mesh**: Integrate with existing enterprise APIs through MCP
- **Real-Time Data Feeds**: Stream live data from business systems
- **Multi-Modal Content**: Access diverse content types through standardized protocols

### 🛡️ **Security & Compliance**
- **Azure Identity Integration**: Enterprise-grade authentication and authorization
- **Secure Connections**: Encrypted communication with MCP servers
- **Access Control**: Role-based permissions for MCP resource access
- **Audit Logging**: Complete traceability of MCP interactions

## ⚙️ Prerequisites & Setup

### 📦 **Required Dependencies**

Install Agent Framework with MCP support:

```bash
pip install agent-framework-azure-ai -U
```

### 🔑 **Azure Configuration**

**Required Azure Resources:**
- Azure AI Foundry workspace
- Appropriate RBAC permissions for AI services
- Network access to MCP endpoints

**Authentication Setup:**
```bash
# Azure CLI authentication
az login
az account set --subscription "your-subscription-id"
```

Let's explore enterprise MCP integrations! 🚀

In [ ]:
# ! pip install -r ../../../../requirements.txt -U

In [ ]:
# 🔌 Import MCP and Azure AI Foundry Components
# Essential components for Model Context Protocol integration with enterprise services

import asyncio
from typing import Any

from agent_framework import Agent, Message
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

from dotenv import load_dotenv

In [ ]:
load_dotenv()

## 🔧 MCP Tool Integration Patterns

This notebook demonstrates **MCP tool integration** with approval workflows:

### 🚀 **Pattern 1: MCP Without Approval**
- MCP tools can be called automatically without user approval
- Best for trusted services and read-only operations
- Fastest execution with no interruptions

### 🔐 **Pattern 2: MCP With Approval Workflow**
- Tool calls require explicit user approval before execution
- Essential for sensitive operations or external API calls
- Demonstrates approval handling with and without sessions

**Key Components:**
- 🔐 **AzureCliCredential**: Secure Azure authentication
- 🔌 **FoundryChatClient**: Foundry-backed chat client with MCP support
- 🏢 **Agent**: Agent Framework's unified agent interface

### Pattern 1: MCP Without Approval - Auto-execute tool calls

In [ ]:
# 🚀 Pattern 1: MCP Without Approval
# MCP tools can be called automatically - no approval required

print("=== MCP without approvals ===")

credential = AzureCliCredential()
client = FoundryChatClient(credential=credential)

# Create MCP tool without approval requirements
mcp_tool = client.get_mcp_tool(
    name="Microsoft Learn MCP",
    url="https://learn.microsoft.com/api/mcp",
)

# Create agent with MCP tool
agent = Agent(
    client=client,
    name="DocsAgent",
    instructions="You are a helpful assistant that can help with microsoft documentation questions.",
    tools=[mcp_tool],
)

# First query
query1 = "How to create an Azure storage account using az cli?"
print(f"User: {query1}")
result1 = await agent.run(query1)
print(f"{agent.name}: {result1.text}\n")

print("\n=======================================\n")

# Second query
query2 = "What is Microsoft Agent Framework?"
print(f"User: {query2}")
result2 = await agent.run(query2)
print(f"{agent.name}: {result2.text}\n")

### Pattern 2: MCP With Approval - Require user approval for specific tools

In [ ]:
# 🔐 Pattern 2: MCP With Approval (Without Session)
# User approval required before tool calls - no session tracking

print("\n=== MCP with approvals (without session) ===")

credential = AzureCliCredential()
client = FoundryChatClient(credential=credential)

# Create MCP tool with specific approval settings
# Never require approval for microsoft_docs_search, but require for others
mcp_tool_with_approval = client.get_mcp_tool(
    name="Microsoft Learn MCP",
    url="https://learn.microsoft.com/api/mcp",
    approval_mode={"never_require_approval": ["microsoft_docs_search"]},
)

# Create agent with approval-required MCP tool
agent_with_approval = Agent(
    client=client,
    name="DocsAgentWithApproval",
    instructions="You answer questions by searching the Microsoft Learn content only.",
    tools=[mcp_tool_with_approval],
)

# Query that will trigger approval workflow
query = "Please summarize the Azure AI Agent documentation related to MCP Tool calling?"
print(f"User: {query}")

# Initial agent run
result = await agent_with_approval.run(query)

# Handle approval requests
while len(result.user_input_requests) > 0:
    new_inputs: list[Any] = [query]
    
    for user_input_needed in result.user_input_requests:
        print(f"\nUser Input Request for function: {user_input_needed.function_call.name}")
        print(f"Arguments: {user_input_needed.function_call.arguments}")
        
        # Add the request to context
        new_inputs.append(Message(role="assistant", contents=[user_input_needed]))
        
        # Get user approval
        user_approval = input("Approve function call? (y/n): ")
        
        # Add approval response
        new_inputs.append(
            Message(
                role="user",
                contents=[user_input_needed.to_function_approval_response(user_approval.lower() == "y")],
            )
        )
    
    # Re-run with approvals
    result = await agent_with_approval.run(new_inputs)

print(f"\n{agent_with_approval.name}: {result.text}")